# Feature Engineering

Load and process the CORINE Land Cover 2018 dataset for Italy, then export a clean station-level feature table for use in `main.ipynb`.

**Dataset:** CORINE Land Cover 2018 (vector, 100 m), Europe — GDB, NUTS: Italia
**Source:** European Environment Agency (EEA)

**Output:** `corine_features.parquet` — one row per Italian air quality station, with land cover features ready to merge into `main.ipynb` on `Air Quality Station EoI Code`.


# CORINE Land Cover 2018

In [ ]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings("ignore")

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'savefig.facecolor': 'white',
    'axes.grid': True,
    'grid.alpha': 0.3
})

## Load CORINE GDB

In [ ]:
GDB_PATH = "LandCover2018/U2018_CLC2018_V2020_20u1.gdb"
LAYER = "U2018_CLC2018_V2020_20u1"

clc = gpd.read_file(GDB_PATH, layer=LAYER)

print("Shape:", clc.shape)
print("CRS:", clc.crs)
print("Columns:", clc.columns.tolist())
clc.head(3)

## Understand the Data Structure

In `Code_18` column there is a 3-level hierarchical code:

- Level 1 - 1st digits only
- Level 2 - 1st + 2nd digits
- Level 3 - All 3 digits


There are 43 unique Level-3 codes, but they all belong to just **5 Level-1 groups**:
- `1xx` - Artificial surfaces (urban, industrial, roads...)
- `2xx` - Agricultural areas
- `3xx` - Forest & semi-natural
- `4xx` - Wetlands
- `5xx` - Water bodies

In [ ]:
# How many unique codes are there?
print("Unique Level-3 codes:", clc["Code_18"].nunique())

print("\nAll codes and how many polygons each covers:")
print(clc["Code_18"].value_counts().to_string())

In [ ]:
# Create Level-1 and Level-2 helper columns by slicing the code string
clc["clc_l1"] = clc["Code_18"].astype(str).str[0]    # e.g. '1'
clc["clc_l2"] = clc["Code_18"].astype(str).str[:2]   # e.g. '11'

# Map Level-1 digit to a readable label
# These 5 groups cover ALL 43 codes — nothing is missed
L1_LABELS = {
    "1": "Artificial surfaces",    # codes 111-142
    "2": "Agricultural areas",     # codes 211-244
    "3": "Forest & semi-natural",  # codes 311-335
    "4": "Wetlands",               # codes 411-422
    "5": "Water bodies"            # codes 511-523
}
clc["clc_l1_label"] = clc["clc_l1"].map(L1_LABELS)

# Verify to receive 0 - meaning every code was recognised
print("Unrecognised codes:", clc["clc_l1_label"].isna().sum())
print("\nPolygons per Level-1 group:")
print(clc["clc_l1_label"].value_counts().to_string())

## Reproject to WGS84

The GDB uses EPSG:3035 (European metric projection). We convert to EPSG:4326 (lat/lon) so it matches the station coordinates in `main.ipynb`.

In [ ]:
clc = clc.to_crs("EPSG:4326")
print("CRS is now:", clc.crs)

## Load Italian Air Quality Stations

We extract unique Italian stations and their coordinates — the same files used in `main.ipynb`.

In [ ]:
# Load all 10 years of EEA data to get station coordinates
DATA = "2015-2024"

frames = []
for year in range(2015, 2025):
    path = os.path.join(DATA, f"DataExtract{year}.csv")
    frames.append(pd.read_csv(path, low_memory=False))

df_main = pd.concat(frames, ignore_index=True)
print(f"Loaded {df_main.shape[0]:,} rows")

In [ ]:
# Keep only Italian stations with valid coordinates, one row per station
stations = (
    df_main[
        (df_main["Country"] == "Italy") &
        df_main["Latitude"].notna() &
        df_main["Longitude"].notna()
    ][["Air Quality Station EoI Code", "Latitude", "Longitude"]]
    .drop_duplicates(subset=["Air Quality Station EoI Code"])
    .reset_index(drop=True)
)

print(f"Unique Italian stations: {len(stations)}")

# Turn into a GeoDataFrame so we can do spatial operations
gdf_stations = gpd.GeoDataFrame(
    stations,
    geometry=gpd.points_from_xy(stations["Longitude"], stations["Latitude"]),
    crs="EPSG:4326"
)
gdf_stations.head(3)

## Assign Land Cover to Each Station

For each station point, we find which CORINE polygon it falls inside.  
Stations on coastlines or near borders that miss the 'within' join get a fallback nearest-polygon assignment.

In [ ]:
# Keep only the columns we need from clc to speed up the join
clc_slim = clc[["Code_18", "clc_l1", "clc_l2", "clc_l1_label", "geometry"]].copy()

# Spatial join: station point -> CORINE polygon it falls within
joined = gpd.sjoin(
    gdf_stations,
    clc_slim,
    how="left",
    predicate="within"
).drop(columns=["index_right"], errors="ignore")

matched = joined["Code_18"].notna().sum()
print(f"Matched: {matched} / {len(joined)} stations ({matched/len(joined)*100:.1f}%)")

In [ ]:
# Fallback for unmatched stations: use nearest polygon instead
unmatched = joined["Code_18"].isna()
print(f"Unmatched stations: {unmatched.sum()}")

if unmatched.sum() > 0:
    fallback = gpd.sjoin_nearest(
        gdf_stations[unmatched],
        clc_slim,
        how="left",
        max_distance=0.05    # ~5 km
    ).drop(columns=["index_right"], errors="ignore")

    fill_cols = ["Code_18", "clc_l1", "clc_l2", "clc_l1_label"]
    joined.loc[unmatched, fill_cols] = (
        fallback.set_index("Air Quality Station EoI Code")[fill_cols]
        .reindex(joined.loc[unmatched, "Air Quality Station EoI Code"])
        .values
    )
    print(f"Still unmatched after fallback: {joined['Code_18'].isna().sum()}")

In [ ]:
# How are stations distributed across land cover categories?
print("Station counts per land cover category \n")

print("Level-1 (broad):")
print(joined["clc_l1_label"].value_counts().to_string())

print("\nLevel-2 urban breakdown:")
print(joined["clc_l2"].value_counts().head(10).to_string())

# Calculate percentages
total = len(joined)
top_cat = joined["clc_l1_label"].value_counts().iloc[0]
top_name = joined["clc_l1_label"].value_counts().index[0]

print(f"\nMost common category: '{top_name}' with {top_cat}/{total} stations ({top_cat/total*100:.1f}%)")


## Feature Decision: Green Space Ratio or Categorical

The distribution check above shows that **76.5% of Italian air quality stations (609/796) sit on Artificial surfaces**.
Using this as a categorical prediction feature for PM2.5 would add almost no value — the model would see
"Artificial surfaces" for 3 out of 4 rows and learn nothing meaningful from it.

Instead, we ask: **"how green is the municipality this station belongs to?"**

For each comune we calculate:

***green_ratio = (area of agricultural + forest) / total area of the comune***


This gives a **continuous variable between 0.0 and 1.0** per station, where:
- `0.0` = fully urban/artificial comune (e.g. dense Milan city centre)
- `1.0` = entirely forested or agricultural comune (e.g. Alpine village)

### Why this is better for prediction
- In our case continuous variable will carry far more information than a dominated categorical one
- It captures the *surrounding environment* of a station, not just the patch directly under it
- Green space ratio is known to correlate with lower PM2.5 and O3 levels through
  vegetation absorption and reduced heat island effects
- It connects naturally to the comuni boundaries already used in `main.ipynb`
  for the missing city enrichment step

## Feature Engineering

We create simple binary (0/1) and categorical features from the CLC codes.  
These will be the **prediction input features** imported into `main.ipynb`.


In [ ]:
# Load comuni boundaries
comuni = gpd.read_file("data_boundaries/comuni_2024/comuni_2024.shp")
comuni = comuni.to_crs("EPSG:4326")

print("Comuni loaded:", comuni.shape)
print("Columns:", comuni.columns.tolist())

In [ ]:
# For each comune, calculate total area and green area

# Green polygons = agricultural (2xx) + forest/semi-natural (3xx)
clc_green = clc_slim[clc_slim["clc_l1"].isin(["2", "3"])].copy()

# Total area per comune
comuni["total_area"] = comuni.geometry.area

# Intersect green CLC polygons with comuni boundaries
green_in_comuni = gpd.overlay(clc_green, comuni[["COMUNE", "geometry"]], how="intersection")
green_in_comuni["green_area"] = green_in_comuni.geometry.area

# Sum green area per comune
green_per_comune = (
    green_in_comuni.groupby("COMUNE")["green_area"]
    .sum()
    .reset_index()
    .rename(columns={"green_area": "total_green_area"})
)

print("Comuni with green area calculated:", len(green_per_comune))

In [ ]:
# Merge and calculate the ratio
comuni = comuni.merge(green_per_comune, on="COMUNE", how="left")

# Comuni with no green polygons at all get 0
comuni["total_green_area"] = comuni["total_green_area"].fillna(0)

# green_ratio: 0.0 = fully urban, 1.0 = fully green
comuni["green_ratio"] = comuni["total_green_area"] / comuni["total_area"]

print("green_ratio summary:")
print(comuni["green_ratio"].describe().round(3))

In [ ]:
# Assign green_ratio to each station via its comune

# Spatial join: which comune does each station fall in?
stations_comuni = gpd.sjoin(
    gdf_stations,
    comuni[["COMUNE", "green_ratio", "geometry"]],
    how="left",
    predicate="within"
).drop(columns=["index_right"], errors="ignore")

matched = stations_comuni["green_ratio"].notna().sum()
print(f"Stations with green_ratio assigned: {matched} / {len(stations_comuni)}")

In [ ]:
# Build the final feature dataframe
df_feats = stations_comuni[[
    "Air Quality Station EoI Code",
    "Latitude", "Longitude",
    "COMUNE", "green_ratio"
]].copy()

print("Final feature table shape:", df_feats.shape)
df_feats.head(200)

In [ ]:
# Check missing values
print("Missing green_ratio values:", df_feats["green_ratio"].isna().sum())

In [ ]:
# Fill the 4 missing with the national median before exporting
df_feats["green_ratio"] = df_feats["green_ratio"].fillna(df_feats["green_ratio"].median())
print("Missing after fill:", df_feats["green_ratio"].isna().sum())

## EXPORT TO MAIN.IPYNB

In [ ]:
OUTPUT = "LandCover2018/green_ratio.csv"
df_feats.to_csv(OUTPUT, index=False)
print(f"Saved {len(df_feats):,} rows - '{OUTPUT}'")
print("Columns:", df_feats.columns.tolist())